# DenseLoRA (symmetric & asymmetric) — TinyLlama-1.1B-Chat di Colab

Menjalankan **DenseLoRA** pada **TinyLlama-1.1B-Chat** dengan dukungan **matriks M non-persegi (asymmetric, r1×r2)** dan tiga aktivasi (**GeLU / SiLU / GoLU**), sesuai Tabel Eksperimen Anda.

| Exp | Aktivasi | r1×r2 | Bentuk M |
|--|--|--|--|
| 1 | GeLU | 32×32 | persegi (baseline) |
| 2 | GeLU | 16×32 | ekspansi |
| 3 | GeLU | 8×16 | ekspansi agresif |
| 4 | SiLU | 32×32 | persegi |
| 5 | SiLU | 16×32 | ekspansi |
| **6** | **SiLU** | **8×16** | **ekspansi agresif** |
| **7** | **GoLU** | **32×32** | **persegi** |
| **8** | **GoLU** | **16×32** | **ekspansi** |
| 9 | GoLU | 8×16 | ekspansi agresif |

Aturan: `alpha = 2 × rank terbesar`, `scaling = alpha / max(r1,r2) = 2.0`.

> **Catatan:** asymmetric (M non-persegi) + GoLU **bukan dari paper** — paper hanya punya M persegi r×r. Ini ekstensi orisinal Anda.
>
> `--adapter_name lora` menjalankan DenseLoRA. Encoder→r1, M:r1→r2, decoder←r2 (encoder/decoder shared antar-layer, M unik per-layer).

**Runtime:** `Runtime → Change runtime type → T4 GPU`.

## 1. Cek GPU

In [ ]:
!nvidia-smi

## 2. Hubungkan Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/DenseLoRA'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)

## 3. Tunjuk folder repo di Drive

Letakkan folder `commonsense_reasoning` (sudah berisi `commonsense_170k.json`, `finetune.py`, dan `peft/` versi baru) di Drive, mis. `MyDrive/DenseLoRA/commonsense_reasoning`. Tidak perlu zip — cukup tunjuk path-nya.

> Opsional (lebih cepat): folder `peft/` berisi banyak file kecil, dan mount Drive lambat membacanya. Set `COPY_TO_LOCAL=True` untuk menyalin folder ke disk lokal Colab (`/content`) sekali di awal — training jadi lebih ngebut. Output tetap disimpan ke Drive (sel 5).

In [ ]:
import os, shutil

# Path folder repo di Drive (sesuaikan bila beda)
REPO_DIR = os.path.join(DRIVE_ROOT, 'commonsense_reasoning')

COPY_TO_LOCAL = False   # True = salin ke /content dulu (baca file lebih cepat saat training)

assert os.path.exists(os.path.join(REPO_DIR, 'finetune.py')), \
    f'finetune.py tidak ada di {REPO_DIR}. Cek lagi lokasi folder di Drive.'
assert os.path.exists(os.path.join(REPO_DIR, 'commonsense_170k.json')), \
    f'commonsense_170k.json tidak ada di {REPO_DIR}.'

if COPY_TO_LOCAL:
    local = '/content/commonsense_reasoning'
    if not os.path.exists(local):
        print('Menyalin ke', local, '... (sekali saja, mohon tunggu)')
        shutil.copytree(REPO_DIR, local)
    REPO_DIR = local

print('REPO_DIR =', REPO_DIR)
# verifikasi patch asymmetric ada
import subprocess
n = subprocess.run(['grep','-c','_make_activation',
                    os.path.join(REPO_DIR,'peft/src/peft/tuners/lora.py')],
                   capture_output=True, text=True).stdout.strip()
print('match _make_activation (harus >=1):', n)

## 4. Install dependencies

In [ ]:
!pip install -q "numpy<2" \
    transformers==4.36.0 accelerate==0.25.0 datasets==2.15.0 \
    sentencepiece==0.1.99 fire==0.5.0 \
    scipy==1.11.4 "huggingface-hub==0.34.4" protobuf
# PENTING: hapus peft versi pip agar finetune.py memakai peft LOKAL (peft/src/) yang berisi DenseLoRA.
# Tanpa ini, `import peft` ke-load peft pip (salah & rusak karena accelerate lama) -> training/eval crash.
!pip uninstall -y peft
# bitsandbytes terbaru: punya binary CUDA 12.x (torch Colab = cu128). Yang lama (0.41.3) crash saat import.
!pip install -q -U bitsandbytes

print('\nSelesai. Jika diminta RESTART, restart lalu ulangi sel 2,3,5.')

## 5. ⚙️ PILIH EKSPERIMEN — ubah di sini

- `EXP`: nomor eksperimen dari tabel (1–9). Default **6**.
- `DATA_PATH`: tempel path file dataset Anda (versi 20% atau 40% yang sudah Anda buat). `DATA_TAG`: label '20'/'40' utk nama folder.
- `MICRO_BATCH`: 32 (turunkan ke 16 bila OOM).

In [ ]:
# ============== PILIH DI SINI ==============
EXP         = 6      # pilih 1..9 (lihat tabel EXPERIMENTS di bawah). Tiap orang ganti angka ini saja.
DATA_PATH   = '/content/drive/MyDrive/DenseLoRA/commonsense_170k_40.json'  # <-- tempel path file 20% / 40% Anda
DATA_TAG    = '40'   # label data utk nama folder output: '20' atau '40'
BATCH       = 64
MICRO_BATCH = 32     # grad-accum = BATCH // MICRO_BATCH = 2
NUM_EPOCHS  = 2
# ==========================================

# Tabel eksperimen: EXP -> (aktivasi, r1, r2). r1=input M, r2=output M (ekspansi bila r2>r1).
EXPERIMENTS = {
    1: ("gelu", 32, 32), 2: ("gelu", 16, 32), 3: ("gelu", 8, 16),
    4: ("silu", 32, 32), 5: ("silu", 16, 32), 6: ("silu", 8, 16),
    7: ("golu", 32, 32), 8: ("golu", 16, 32), 9: ("golu", 8, 16),
}
ACT, R1, R2 = EXPERIMENTS[EXP]
EFF_RANK = max(R1, R2)        # dikirim ke --lora_r (param-count & guard r>0)
ALPHA    = 2 * max(R1, R2)    # alpha = 2 x rank terbesar -> scaling = 2.0

import os
assert os.path.exists(DATA_PATH), f'Dataset tidak ada: {DATA_PATH}  (cek path / nama file di Drive)'
assert BATCH % MICRO_BATCH == 0

OUTPUT_DIR = os.path.join(
    DRIVE_ROOT,
    f"denselora_tinyllama_exp{EXP}_{ACT}_{R1}x{R2}_d{DATA_TAG}"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
shape = "persegi" if R1==R2 else ("ekspansi" if R2>R1 else "kompresi")
print(f"EXP {EXP}: aktivasi={ACT.upper()}  M={R1}x{R2} ({shape})  alpha={ALPHA}  scaling={ALPHA/max(R1,R2)}")
print(f"dataset={DATA_PATH}")
print(f"data_tag={DATA_TAG}%  bs={BATCH}  mbs={MICRO_BATCH}  grad_accum={BATCH//MICRO_BATCH}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

## 6. (Opsional) Cek jumlah sampel

In [ ]:
import json, os
n = len(json.load(open(DATA_PATH)))
print(f'File: {DATA_PATH}')
print(f'Jumlah sampel di file ini: {n:,}  (dipakai semua, file sudah dikurangi sendiri)')

## 7. Fine-tuning — auto-save Drive + auto-resume

Folder output unik per-eksperimen, jadi resume tidak tertukar antar-exp. Bila terputus, jalankan ulang sel 2,3,5,7.

In [ ]:
import os
os.chdir(REPO_DIR)
os.environ['WANDB_DISABLED'] = 'true'   # matikan prompt wandb
os.environ.update({k:str(v) for k,v in dict(
    OUTPUT_DIR=OUTPUT_DIR, DATA_PATH=DATA_PATH, BATCH=BATCH,
    MICRO_BATCH=MICRO_BATCH, NUM_EPOCHS=NUM_EPOCHS,
    EFF_RANK=EFF_RANK, R1=R1, R2=R2, ALPHA=ALPHA, ACT=ACT,
).items()})
print('cwd =', os.getcwd())

In [ ]:
!python finetune.py \
    --base_model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' \
    --data_path "$DATA_PATH" \
    --output_dir "$OUTPUT_DIR" \
    --batch_size $BATCH --micro_batch_size $MICRO_BATCH --num_epochs $NUM_EPOCHS \
    --learning_rate 3e-4 --cutoff_len 256 --val_set_size 120 \
    --eval_step 80 --save_step 80 \
    --adapter_name lora \
    --target_modules '["q_proj","k_proj","v_proj","up_proj","down_proj"]' \
    --lora_r $EFF_RANK --lora_r1 $R1 --lora_r2 $R2 \
    --lora_alpha $ALPHA --dense_activation $ACT \
    --use_gradient_checkpointing \
    --auto_resume True

### Cek checkpoint di Drive (opsional)

In [ ]:
import os
ck=sorted([d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')],
          key=lambda x:int(x.split('-')[1])) if os.path.isdir(OUTPUT_DIR) else []
print('Checkpoint:', ck)
if ck: print('Isi terakhir:', os.listdir(os.path.join(OUTPUT_DIR, ck[-1])))

## 8. (Opsional) Evaluasi commonsense
Eval otomatis memakai r1/r2/aktivasi dari `adapter_config.json`, jadi shape M cocok.

In [ ]:
import os
os.chdir(REPO_DIR)
if not os.path.isdir('dataset'):
    !git clone --depth 1 https://github.com/AGI-Edgerunners/LLM-Adapters.git /tmp/llm_adapters
    !cp -r /tmp/llm_adapters/dataset ./dataset
os.makedirs('experiment', exist_ok=True)
print('dataset:', sorted(os.listdir('dataset'))[:10])

In [ ]:
import os
os.environ['LW']=OUTPUT_DIR
for ds in ['boolq','piqa','social_i_qa']:
    print('==== Evaluating', ds, '===='); os.environ['DS']=ds
    !python commonsense_evaluate.py \
        --model LLaMA3-8B --adapter LoRA --dataset "$DS" \
        --base_model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' \
        --batch_size 1 --lora_weights "$LW" | tee -a "$LW/$DS.txt" 

## 9. 📊 Ringkasan trade-off semua eksperimen

Sel ini memindai semua folder `denselora_tinyllama_exp*` di Drive, lalu mengumpulkan:
- **trainable_params** (dihitung dari bobot adapter tersimpan = Encoder + Decoder + M),
- **param_%** terhadap 1.1B,
- **akurasi tiap task** + **rata-rata** (di-parse dari file `*.txt` hasil evaluasi).

Hasilnya ditampilkan sebagai tabel dan disimpan ke `ringkasan_eksperimen.csv` di Drive. Jalankan **setelah** training+eval beberapa exp (mis. 6, 7, 8).

In [ ]:
import os, re, glob, csv
# Catatan: TIDAK meng-import pandas di awal supaya tahan terhadap bentrok ABI numpy
# (numpy<2 vs pandas bawaan Colab). Tabel dibuat dengan Python biasa + CSV.
try:
    import torch
except Exception:
    torch = None

BASE_PARAMS = 1.1e9  # TinyLlama-1.1B

def count_trainable_params(run_dir):
    cands = [os.path.join(run_dir, fn) for fn in ('adapter_model.bin', 'pytorch_model.bin')]
    cks = sorted(glob.glob(os.path.join(run_dir, 'checkpoint-*')),
                 key=lambda x: int(x.split('-')[-1]))
    for ck in reversed(cks):
        cands.append(os.path.join(ck, 'pytorch_model.bin'))
    for p in cands:
        if os.path.exists(p) and torch is not None:
            try:
                sd = torch.load(p, map_location='cpu')
                if isinstance(sd, dict) and 'state_dict' in sd:
                    sd = sd['state_dict']
                return int(sum(v.numel() for v in sd.values() if hasattr(v, 'numel')))
            except Exception as e:
                print('warn baca', p, e)
    return None

ACC_RE = re.compile(r'accuracy\s+\d+\s+([0-9.]+)')
def parse_accuracies(run_dir):
    out = {}
    for txt in glob.glob(os.path.join(run_dir, '*.txt')):
        ds = os.path.splitext(os.path.basename(txt))[0]
        try:
            s = open(txt, errors='ignore').read()
        except Exception:
            continue
        ms = ACC_RE.findall(s)
        if ms:
            out[ds] = round(float(ms[-1]), 4)
    return out

FOLDER_RE = re.compile(r'denselora_tinyllama_exp(\d+)_([a-z]+)_(\d+)x(\d+)_d(\d+)')
rows, task_cols = [], []
for d in sorted(glob.glob(os.path.join(DRIVE_ROOT, 'denselora_tinyllama_exp*'))):
    m = FOLDER_RE.search(os.path.basename(d))
    if not m:
        continue
    exp, act, r1, r2, frac = m.groups()
    params = count_trainable_params(d)
    accs = parse_accuracies(d)
    avg = round(sum(accs.values()) / len(accs), 4) if accs else None
    row = dict(exp=int(exp), aktivasi=act.upper(), M=f'{r1}x{r2}',
               bentuk=('persegi' if r1 == r2 else ('ekspansi' if int(r2) > int(r1) else 'kompresi')),
               data=int(frac), trainable_params=params,
               param_pct=(round(100 * params / BASE_PARAMS, 4) if params else None),
               avg_acc=avg)
    for k in accs:
        if k not in task_cols:
            task_cols.append(k)
    row.update(accs)
    rows.append(row)

rows.sort(key=lambda r: r['exp'])
cols = ['exp', 'aktivasi', 'M', 'bentuk', 'data', 'trainable_params', 'param_pct', 'avg_acc'] + task_cols

if not rows:
    print('Belum ada folder eksperimen denselora_tinyllama_exp*. Jalankan training (+eval) dulu.')
else:
    def fmt(v):
        return '' if v is None else str(v)
    widths = {c: max(len(c), max(len(fmt(r.get(c))) for r in rows)) for c in cols}
    header = ' | '.join(c.ljust(widths[c]) for c in cols)
    print(header)
    print('-' * len(header))
    for r in rows:
        print(' | '.join(fmt(r.get(c)).ljust(widths[c]) for c in cols))

    out_csv = os.path.join(DRIVE_ROOT, 'ringkasan_eksperimen.csv')
    with open(out_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            w.writerow({c: r.get(c) for c in cols})
    print('\nDisimpan ke:', out_csv)

    # Tampilan tabel cantik HANYA jika pandas bisa di-import tanpa error ABI
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows)[cols])
    except Exception as e:
        print('(pandas tidak dipakai untuk tampilan:', type(e).__name__, '- tabel teks di atas tetap lengkap)')

## Menjalankan beberapa eksperimen (6, 7, 8, ...)
Untuk tiap eksperimen: ubah `EXP` di sel 5 → jalankan sel 7. Tiap exp punya `OUTPUT_DIR` sendiri, jadi checkpoint & resume tidak bertabrakan. Contoh urutan: `EXP=6` → train → `EXP=7` → train → `EXP=8` → train.

### Catatan
- **Asymmetric/GoLU bukan dari paper.** TinyLlama 1.1B + data 20/40% → akurasi tidak mereproduksi angka paper; ini untuk membandingkan antar-konfigurasi Anda sendiri (efek aktivasi & bentuk M), bukan replikasi.
- **OOM (mbs=32):** turunkan `MICRO_BATCH` ke 16 — konsisten sejak awal satu run (mengubah mbs di tengah merusak konsistensi resume).
- **GoLU** dihitung di fp32 lalu di-cast fp16 untuk stabilitas (sudah ditangani di `lora.py`).